Aprendizado de variedades (Manifold Learning) com fluxo de clusterização distribuída por tipo de característica é mais informativo do que o UMAP para conjuntos de dados clínicos tabulares

Importando as Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
import math
import umap.umap_ as umap
%config InlineBackend.figure_format = 'svg'

In [ ]:
from cluster_val import *

Importando os dados

In [ ]:
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
data_with_target=pd.read_csv('preprocessed_migraine_data.csv')

In [ ]:
np.random.seed(42)
data=data_with_target.sample(frac=1) #Emabaralha o dataset
np.random.seed(42)
i=[x for x in range(400)]

data.set_index(pd.Series(i), inplace=True)

In [ ]:
data.drop(['Unnamed: 0','Type'],axis=1,inplace=True)

UMAP nos dados originais

In [ ]:
from fdc.fdc import feature_clustering

In [ ]:
umap_emb=feature_clustering(15,0.1,'euclidean',data,True)

Silhouette_score e Dunn index para os clusters do UMAP extraidos usando clustering K-means

In [ ]:
from fdc.clustering import Clustering

umap_clustering=Clustering(umap_emb,umap_emb,True)
umap_cluster_list,umap_cluster_counts=umap_clustering.K_means(2)

In [ ]:
from sklearn import metrics
from sklearn.metrics import pairwise_distances
from sklearn.metrics import silhouette_score

In [ ]:
silhouette_score(umap_emb, umap_cluster_list, metric='euclidean')

Visualizar Silhouette score (Pode-se escolher o números de clusters baseados no resultado)

In [ ]:
Silhouette_visual(umap_emb)

Elbow plot para o embedding

In [ ]:
elbow_plot(umap_emb)

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list))

Silhouette_score e Dunn index para os clusters UMAP extraídos usando clustering aglomerativo

In [ ]:
umap_cluster_list_agglo,umap_cluster_counts_agglo=umap_clustering.Agglomerative(2,'euclidean','ward')

In [ ]:
silhouette_score(umap_emb, umap_cluster_list_agglo, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list_agglo))

Silhouette_score e Dunn index para os clusters UMAP extraídos usando clustering do DBSCAN

In [ ]:
umap_cluster_list_dbscan,umap_cluster_counts_dbscan=umap_clustering.DBSCAN(0.5,20)

In [ ]:
#Remove ruído
non_noise_indices= np.where(np.array(umap_cluster_list_dbscan)!=-1)
umap_emb= umap_emb.iloc[non_noise_indices]
#FDC_emb_low= FDC_emb_low.iloc[non_noise_indices]
umap_cluster_list_dbscan= np.array(umap_cluster_list_dbscan)[non_noise_indices]

In [ ]:
silhouette_score(umap_emb, umap_cluster_list_dbscan, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list_dbscan))

Dividindo as variaveis
- cont_list = continuas
- ord_list = ordinais
-  nom_list  = nominais

In [ ]:
cont_list=['Age']

ord_list=['Duration','Frequency','Intensity','Visual','Nausea', 'Vomit']

nom_list=['Location','Character','Dysphasia', 'Vertigo', 'Tinnitus','Sensory','Phonophobia','Photophobia', 'Hypoacusis', 'Diplopia', 'Defect','Conscience', 'Paresthesia', 'DPF']

In [ ]:
len(ord_list)

In [ ]:
len(nom_list)

In [ ]:
len(cont_list)

FDC nos dados originais

In [ ]:
from fdc.fdc import FDC, Clustering
from fdc.fdc import canberra_modified
modified_can = canberra_modified

In [ ]:
fdc = FDC(clustering_cont=Clustering('euclidean',15,0.1,max_components=1)
          , clustering_ord=Clustering(modified_can,15,0.1)
          , clustering_nom=Clustering('hamming',15,0.1)
          , visual=True
          , use_pandas_output=True
          , with_2d_embedding=True
          )

fdc.selectFeatures(continueous=cont_list, nomial=nom_list, ordinal=ord_list)

FDC_emb_high,FDC_emb_low = fdc.normalize(data,n_neighbors=15, min_dist=0.1,cont_list=cont_list, nom_list=nom_list, ord_list=ord_list,
                  with_2d_embedding=True,
                  visual=True)

Silhouette_score e Dunn index para os clusters FDC (de dimensão intermediaria) extraídos usando k-means

In [ ]:
from fdc.clustering import Clustering

clustering=Clustering(FDC_emb_high,FDC_emb_low,True)
cluster_list,cluster_counts=clustering.K_means(7)

In [ ]:
FDC_emb_high['Cluster'] = cluster_list

In [ ]:
silhouette_score(FDC_emb_high, cluster_list, metric='euclidean')

In [ ]:
Silhouette_visual(FDC_emb_high)

In [ ]:
elbow_plot(FDC_emb_high)

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_high,cluster_list))

Silhouette_score e Dunn index para os clusters FDC (de dimensão intermediaria) extraídos usando clustering aglomerativo

In [ ]:
FDC_emb_high['Cluster'] = cluster_list_agglo

In [ ]:
silhouette_score(FDC_emb_high, cluster_list_agglo, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_high,cluster_list_agglo))

Silhouette_score e Dunn index para os clusters FDC (de dimensão intermediaria) extraídos usando clustering do DBSCAN

In [ ]:
cluster_list_dbscan,cluster_counts_dbscan=clustering.DBSCAN(1.5,15)

In [ ]:
cluster_counts_dbscan

In [ ]:
FDC_emb_high['Cluster'] = cluster_list_dbscan

In [ ]:
#Remove ruído
non_noise_indices= np.where(np.array(cluster_list)!=-1)
FDC_emb_high= FDC_emb_high.iloc[non_noise_indices]
FDC_emb_low= FDC_emb_low.iloc[non_noise_indices]
cluster_list= np.array(cluster_list)[non_noise_indices]

In [ ]:
silhouette_score(FDC_emb_high, cluster_list_dbscan, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_high,cluster_list_dbscan))